# Token Probe — mesure directe via Kong

Script isolé : appelle Kong directement (sans LangGraph) et lit `usage` dans la réponse.
Permet de mesurer exactement combien de tokens consomme un prompt donné.

In [ ]:
import json
import os
from pathlib import Path

import httpx
from dotenv import load_dotenv

# Charge le .env depuis la racine du projet (fonctionne que le notebook
# soit lancé depuis scripts/ ou depuis la racine)
for candidate in [Path(".env"), Path("../.env")]:
    if candidate.exists():
        load_dotenv(candidate, override=True)
        print(f".env charge depuis : {candidate.resolve()}")
        break

KONG_CHAT_URL = os.environ["KONG_CHAT_URL"]
KONG_API_KEY  = os.environ.get("KONG_API_KEY", "dummy")
GEMINI_MODEL  = os.environ.get("GEMINI_MODEL", "gemini-3.5-flash")

print(f"Kong URL  : {KONG_CHAT_URL}")
print(f"Modele    : {GEMINI_MODEL}")

In [ ]:
import sys
sys.path.insert(0, str(Path("../src")))

from agentic4api.graph.prompts import SYSTEM_PROMPT, SYSTEM_PROMPT_BT, SYSTEM_PROMPT_V2

# ── Prompt à tester ────────────────────────────────────────────────────────────
# Choisis le prompt à mesurer en changeant la variable ci-dessous :
#   SYSTEM_PROMPT    → mode agentic text (SEARCH:) — celui utilisé en prod
#   SYSTEM_PROMPT_BT → mode bind_tools (sans SEARCH:)
#   SYSTEM_PROMPT_V2 → variante phase 3 (non activée)

ACTIVE_PROMPT = SYSTEM_PROMPT

USER_MESSAGE = "Comment créer une commande client ?"

# Contexte Pinecone simulé (optionnel — laisse vide pour tester sans contexte)
PINECONE_CONTEXT = """
- name: order-api-v4 | statut: active | description: Gestion des commandes e-commerce.
- name: order-api-v3 | statut: deprecated | description: Version précédente.
"""

print(f"Prompt actif : {len(ACTIVE_PROMPT)} caracteres (~{len(ACTIVE_PROMPT)//4} tokens estimés)")

In [ ]:
def call_kong(messages: list[dict], max_tokens: int = 1024) -> dict:
    """Appel direct à Kong — retourne le JSON brut de la réponse."""
    payload = {
        "model": GEMINI_MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": 0.0,
    }
    headers = {
        "Content-Type": "application/json",
        # Authorization supprimé volontairement : Kong gère l'auth vers Google en interne
    }
    with httpx.Client(verify=False, timeout=60) as client:
        resp = client.post(KONG_CHAT_URL, json=payload, headers=headers)
        resp.raise_for_status()
        return resp.json()


def parse_usage(response: dict) -> dict:
    """Extrait les tokens depuis usage ou usage_metadata selon le format retourné."""
    usage = response.get("usage") or {}
    t_in    = usage.get("prompt_tokens")     or usage.get("input_tokens", 0)
    t_out   = usage.get("completion_tokens") or usage.get("output_tokens", 0)
    t_total = usage.get("total_tokens", t_in + t_out)
    t_think = max(0, t_total - t_in - t_out)
    return {
        "tokens_in":    t_in,
        "tokens_out":   t_out,
        "tokens_think": t_think,
        "tokens_total": t_total,
        "usage_raw":    usage,
    }

In [ ]:
# ── Test 1 : question seule (sans contexte Pinecone) ───────────────────────────

messages_sans_contexte = [
    {"role": "system",  "content": ACTIVE_PROMPT},
    {"role": "user",    "content": USER_MESSAGE},
]

resp1   = call_kong(messages_sans_contexte)
usage1  = parse_usage(resp1)
answer1 = resp1["choices"][0]["message"]["content"]

print("=== Sans contexte Pinecone ===")
print(f"  Tokens IN    : {usage1['tokens_in']}")
print(f"  Tokens OUT   : {usage1['tokens_out']}")
print(f"  Tokens THINK : {usage1['tokens_think']}")
print(f"  Tokens TOTAL : {usage1['tokens_total']}")
print(f"  Usage brut   : {usage1['usage_raw']}")
print(f"\nRéponse : {answer1[:200]}...")

In [ ]:
# ── Test 2 : avec contexte Pinecone injecté ────────────────────────────────────

messages_avec_contexte = [
    {"role": "system",    "content": ACTIVE_PROMPT},
    {"role": "user",      "content": USER_MESSAGE},
    {"role": "assistant", "content": "SEARCH: creer commande client"},
    {"role": "user",      "content": f"[Résultats Pinecone]\n{PINECONE_CONTEXT}"},
]

resp2   = call_kong(messages_avec_contexte)
usage2  = parse_usage(resp2)
answer2 = resp2["choices"][0]["message"]["content"]

print("=== Avec contexte Pinecone ===")
print(f"  Tokens IN    : {usage2['tokens_in']}")
print(f"  Tokens OUT   : {usage2['tokens_out']}")
print(f"  Tokens THINK : {usage2['tokens_think']}")
print(f"  Tokens TOTAL : {usage2['tokens_total']}")
print(f"  Usage brut   : {usage2['usage_raw']}")
print(f"\nRéponse : {answer2[:200]}...")

In [ ]:
# ── Comparaison ────────────────────────────────────────────────────────────────

print("=" * 50)
print(f"{'Métrique':<20} {'Sans contexte':>15} {'Avec contexte':>15} {'Delta':>10}")
print("=" * 50)
for key in ("tokens_in", "tokens_out", "tokens_think", "tokens_total"):
    v1 = usage1[key]
    v2 = usage2[key]
    print(f"{key:<20} {v1:>15} {v2:>15} {v2-v1:>+10}")
print("=" * 50)
print()
print("Cout estimé (Gemini 2.5 Flash) :")
for label, usage in [("Sans contexte", usage1), ("Avec contexte", usage2)]:
    cost = (
        usage["tokens_in"]    * 0.15  / 1_000_000 +
        usage["tokens_out"]   * 0.60  / 1_000_000 +
        usage["tokens_think"] * 3.50  / 1_000_000
    )
    print(f"  {label:<20} : ${cost:.6f}")

In [ ]:
# ── Inspection du JSON brut complet (debug) ───────────────────────────────────
print("JSON brut (dernier appel) :")
print(json.dumps({k: v for k, v in resp2.items() if k != "choices"}, indent=2))